<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/torneos/notebooks/c6_examen.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C6-Examen · Entrega guiada: Quant de torneos verificado
Parte 1 walk-forward honesto en BTC (lags + embargo 2, 3 folds, costos 4 bps) · Parte 2 submission de torneo por eras · Parte 3 bitácora de 20 trades con kill-switch.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/torneos/data/c6_examen.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c6_examen.csv'), Path('data/c6_examen.csv'), Path('c6_examen.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
PARTE1 = 'features sin leakage: solo lags del close y volumen (high actual prohibido)'
df['ret'] = df['close'].pct_change()
for k in (1, 2, 3, 5):
    df['lag_%d' % k] = df['ret'].shift(k)
df['mom10'] = df['close'] / df['close'].shift(10) - 1
df['vol_z'] = (df['volumen'] - df['volumen'].rolling(10).mean()) / df['volumen'].rolling(10).std()
EMBARGO = 2
data = df.dropna().reset_index(drop=True)
data['y'] = (data['ret'].shift(-1) > 0).astype(int)
data = data.iloc[:-1].reset_index(drop=True)
feat = ['lag_1', 'lag_2', 'lag_3', 'lag_5', 'mom10', 'vol_z']
assert 'high' not in feat and 'low' not in feat
print('filas:', len(data), 'features:', feat)

In [ ]:
import numpy as np
from sklearn.linear_model import RidgeClassifier
X, y = data[feat].values, data['y'].values
n = len(X)
bounds = [(n - 54, n - 36), (n - 36, n - 18), (n - 18, n)]
sharpes, rets_all = [], []
for (c, d) in bounds:
    m = RidgeClassifier().fit(X[:c - EMBARGO], y[:c - EMBARGO])
    sig = np.where(m.predict(X[c:d]) == 1, 1.0, -1.0)
    r = sig * data['ret'].values[c:d] - 0.0004
    rets_all.append(r)
    sharpes.append(float(r.mean() / (r.std() + 1e-12) * np.sqrt(252)))
rets_all = np.concatenate(rets_all)
sharpe_oos = float(rets_all.mean() / (rets_all.std() + 1e-12) * np.sqrt(252))
print('Sharpe por fold (3 folds):', [round(s, 2) for s in sharpes])
print('Sharpe OOS neto (honesto, aunque sea <= 0): %.2f' % sharpe_oos)

In [ ]:
PARTE2 = 'submission por eras: 6 eras de 20 dias, Spearman crudo/neutral, t-stat'
from scipy.stats import spearmanr
idx = np.arange(len(data))
data['pred'] = 0.0
for e in range(6):
    mask = (idx // 20) == e
    Xi, yi = X[mask], y[mask]
    data.loc[mask, 'pred'] = RidgeClassifier().fit(Xi, yi).decision_function(Xi)
rows = []
for e in range(6):
    g = data[(idx // 20) == e]
    fut = g['ret'].shift(-1).fillna(0)
    crudo, _ = spearmanr(g['pred'], fut)
    beta = g['pred'].cov(g['vol_z']) / (g['vol_z'].var() + 1e-12)
    resid = g['pred'] - beta * g['vol_z']
    fnc, _ = spearmanr(resid, fut)
    rows.append((e + 1, float(crudo), float(fnc)))
sub = pd.DataFrame(rows, columns=['era', 'spearman', 'spearman_neutral'])
t = sub.spearman_neutral.mean() / (sub.spearman_neutral.std(ddof=1) / np.sqrt(len(sub)) + 1e-12)
print(sub.round(4).to_string(index=False))
print('t-stat neutral: %.2f' % t)
submission = pd.DataFrame({'id': ['btc_%03d' % i for i in range(len(data))], 'prediction': data['pred'].rank(pct=True).round(4)})
print(submission.head(3).to_string(index=False))

In [ ]:
PARTE3 = 'bitacora de 20 trades con riesgo 1% y kill-switch -3%'
sig_all = np.where(RidgeClassifier().fit(X, y).predict(X) == 1, 1.0, -1.0)
R_log, dia_pnl = [], []
for i in range(20):
    j = min(i * 5 + 1, len(data) - 1)
    r = float(sig_all[i * 5] * data['ret'].values[j] * 100)
    R_log.append({'trade': i + 1, 'setup': 'momentum_era', 'R': round(r, 2), 'emocion': 'calmado', 'siguio_plan': 1})
    dia_pnl.append(r / 10)
bit = pd.DataFrame(R_log)
kills = sum(1 for p in dia_pnl if p <= -3.0)
print('expectancy 20 trades: %+.2fR · kill-switch activaciones: %d' % (bit.R.mean(), kills))
assert len(bit) == 20 and (bit.siguio_plan == 1).all()
assert np.isfinite(sharpe_oos) and list(submission.columns) == ['id', 'prediction']
assert ((submission.prediction >= 0) & (submission.prediction <= 1)).all()
print('OK EXAMEN: WF + submission id,prediction + bitacora 20 trades verificados')